In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Analyze redshift and mean fluxes of SDSS spectra

In [ ]:
def load_spectra_cache(path):
    """Load a spectra cache saved by build_and_save_spectra. Returns a dict of arrays."""
    with np.load(path) as npz:
        return {
            "wavelength": npz["wavelength"],
            "flux": npz["flux"],
            "mask": npz["mask"],
            "snr": npz["snr"],
            "redshift": npz["redshift"],
            "pmf": npz["pmf"],
            "valid_spectrum": npz["valid_spectrum"],
            "z_min": float(npz["z_min"]),
            "z_max": float(npz["z_max"]),
        }

In [ ]:
out_file_path = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/SDSS_spectra/SDSS_support_files/spectra_cache.npz"

data = load_spectra_cache(out_file_path)

### Plot example spectrum

In [ ]:
fig = plt.figure(figsize=(10, 3))

plt.plot(data["wavelength"], data["flux"][0], label="spectrum")
plt.plot(data["wavelength"], [-0.5 if x else 0 for x in data["mask"][0]], label="mask")
plt.xlabel("Wavelength (Å)")
plt.ylabel("Flux")
plt.legend()
plt.show()

### Make Quasar redshift histogram

In [ ]:
redshifts = data["redshift"]

fig, ax = plt.subplots(figsize=(4,4))

ax.hist(redshifts, bins=60, histtype='step', linewidth=2, color='r', range=(2, 5))

ax.set_xlabel('Redshift, z')
ax.set_ylabel('Number of Quasars in SDSS DR9 Catalogue')

# Main axis: redshift
ax.set_xlim(2, 5)
ax.grid()

# Secondary axis: wavelength
def redshift_to_wavelength(z):
    return 1215.67 * (z + 1)

def wavelength_to_redshift(wavelength):
    return wavelength / 1215.67 - 1

secax = ax.secondary_xaxis('top', functions=(redshift_to_wavelength, wavelength_to_redshift))
secax.set_xlabel(r'Quasar Ly$\alpha$ Peak Wavelength ($\AA$)')

plt.tight_layout()
plt.savefig('plots/lyman_alpha_peak_hist.pdf', format='pdf')
plt.show()

### Compute bounds by Lyman Beta line

In [ ]:
# for higher redshifts the ly beta line gets shifted into the lower wavelength range. Thus this creates a upper bound on what redshifts we can include for a given lower wavelngth edge. -QUESTION: is lyman beta even a probelm? It should also be in the reals spectra, right? Just need to be careful when combining the spectra then i guess and add the absorptions?
print(data["z_min"])
print(data["z_max"])

lower_edge = 3800

upper_ly_beta_bound = 1215.67 * (lower_edge/1025.7220)

print("Given lower edge in A", lower_edge)
print("Upper bound in A", upper_ly_beta_bound)
print("Upper bound in z", (upper_ly_beta_bound/1215.67) - 1)

## Make mean flux plot

### Reference data from 

In [ ]:
# Data from https://arxiv.org/abs/0709.2382v3 

kirkman_metal_corr = [
    0.125, 0.109, 0.154, 0.213, 0.170, 0.228,
    0.262, 0.281, 0.335, 0.346, 0.413
]

sigma_tau_eff_tot = [
    0.027, 0.022, 0.023, 0.026, 0.018, 0.022,
    0.024, 0.026, 0.030, 0.029, 0.035
]

z_paper = [
    2.0, 2.1, 2.2, 2.3, 2.4, 2.5,
    2.6, 2.7, 2.8, 2.9, 3.0]


tau = np.array(kirkman_metal_corr)

# Flux
F = np.exp(-tau)

# Exact asymmetric error propagation
F_low = np.exp(-(tau + sigma_tau_eff_tot))
F_high = np.exp(-(tau - sigma_tau_eff_tot))

# Error-bar sizes
dF_minus = F - F_low
dF_plus = F_high - F


### Simulated box values

In [ ]:
import h5py
import glob
import temet
import os


def load_spectra_from_boxes(gp_paths, spectrumNums, y_param="flux", z=2.0):
    """ Loads all spectra specified in spectrumNums for all boxes specified in gp_paths

    Args:
        gp_paths (list): List of Gridpoint folder paths to load spectra from.
        spectrumNums (list): List of indices for the spectra.
        y_param (str, optional): name of field to return data from. Defaults to 'flux'.
        z (float, optional): Redshift value. Defaults to 2.0.

    Returns:
        wavelengths (list): one np.array containing the wavelengths of the spectra for each box
        y_param (list): len(spectrumNums) number of np.arrays for each box (each is one spectrum)
        params (list): [Omega0, OmegaBaryon, OmegaLambda, HubbleParam] for each box
    """
    
    wavelengths = []
    y_out = []
    #params = []
    
    for gp_path in gp_paths:
        gp_name = gp_path.split("/")[-2]
        path = gp_path + f"data.files/spectra/spectra_{gp_name}_z{z:.1f}_n100d2-fullbox_SDSS-BOSS_HI_combined.hdf5"
        #print(path)
        with h5py.File(path, "r") as f:     
            # Zugriff auf einzelne Datasets
            #print(f.keys())  # Gibt die Namen der Datasets im HDF5-File aus
            wavelengths.append(f["wave"][:])  # Das [:] liest das gesamte Dataset in ein NumPy-Array ein
            y_out.append(f[y_param][:][spectrumNums])

        #Omega0, OmegaBaryon, OmegaLambda, HubbleParam = get_cosmo_parameters(gp_path)

        #params.append([Omega0, OmegaBaryon, OmegaLambda, HubbleParam])

    return wavelengths, y_out#, params


def redshift_wavelength_forward(z, wavelength):
    """ Redshifts a wavelength forward (wavelength gets bigger with more redshift)
    
    Args:
        z (float): redshift to apply to wavelength
        wavelength (float or np.array): wavelength to be shifted

    Returns:
        (float or np.array): redshifted wavelength
    """
    return (z+1)*wavelength

def get_data_from_header(output_dir, snapN, param='BoxSize'):
    """ get data from the header of a specified simulation and snapshot
    
    Args:
        output_dir (string): base directory of the simulation
        snapN (int): number of the snapshot
        param (string): parameter of the hdf5 file (the parameter to read)

    Returns:
        np.array : Array containing the specific data
    """

    snapdir = glob.glob(output_dir+f"output/snapdir_*{snapN}")[0]
    snap_files = os.listdir(snapdir)
    file_name = snap_files[0]
    file_path = snapdir+f"/{file_name}"

    with h5py.File(file_path, "r") as f:
        header = f['Header']
        return_param = header.attrs[param]
        
    return return_param

def get_edges(gp_path, snapN=12, redshift=2.0):
    Ly_alpha_0 = 1215.67 

    sim = temet.sim(gp_path, redshift=redshift)
    dz = sim.dz
    z = get_data_from_header(gp_path, snapN, param="Redshift")
    z = z + dz
    print(dz)
    right_edge = redshift_wavelength_forward(z, Ly_alpha_0)
    left_edge = redshift_wavelength_forward(z - dz, Ly_alpha_0)  # well this is awkward and ugly code (this really is just z + dz - dz)

    return left_edge, right_edge

In [ ]:
sim_suite_location = "/pfs/10/project/bw21g005/ly_alpha_sbi_paper/L50n512_suite/reference/"
gps_to_reorder = []
gps_to_reorder.append(sim_suite_location)


redshifts_sims = [2.0, 2.1, 2.2, 2.3, 2.4, 2.5 , 2.6, 2.7, 2.8, 2.9, 3.0]
snaps = [12, 11, 10, 9, 8, 7, 6, 5, 4, 3, "002"]
mean_fluxes_sims = []
std_fluxes_sims = []

for i, z in enumerate(redshifts_sims):
    print(z)
    w, y = load_spectra_from_boxes(gps_to_reorder, spectrumNums=[i for i in range(10000)], z=z)
    fluxes = np.array(y[0])
    l, r = get_edges(gps_to_reorder[0], snapN=snaps[i], redshift=z)
    fluxes = fluxes[:, (w[0] >= l) & (w[0] <= r)]

    mean_flux = fluxes.mean(axis=1).mean()
    std_flux = fluxes.mean(axis=1).std()

    mean_fluxes_sims.append(mean_flux)
    std_fluxes_sims.append(std_flux)

    print("mean", mean_flux)
    print("std", std_flux)

In [ ]:
print(data["redshift"].min(), data["redshift"].max())

redshift_bins = [(1.95, 2.0), (2.05, 2.1), (2.15, 2.2), (2.25, 2.3), (2.35, 2.4), (2.45, 2.5), (2.55, 2.6), (2.65, 2.7), (2.75, 2.8), (2.85, 2.9), (2.95, 3.0)]
LYA_REST = 1215.67

In [ ]:
def plot_mean_flux(result, ref_F, ref_z, ref_minus, ref_plus, sdss_zs, sdss_mean_fluxes, sdss_errs, save=False):
    z_centers = [zhi-0.02 for zlo, zhi in result["z_edges"]]
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.errorbar(
        z_centers, result["mean_flux"],
        yerr=result["sem"], 
        fmt="s", 
        capsize=5,
        label="SDSS spectra",
        color='C0',
        barsabove=True,
        elinewidth=2,
        markeredgewidth=2,
        zorder=7
    )

    plt.errorbar(
        sdss_zs, sdss_mean_fluxes,
        yerr=sdss_errs,
        fmt='s',
        capsize=5,
        label="Simulated reference box",
        color='C1',
        barsabove=True,
        elinewidth=2,
        markeredgewidth=2,
        zorder=5
    )

    ax.errorbar(
        np.array(ref_z)+0.02, ref_F,
        yerr=[ref_minus, ref_plus],
        fmt='s',
        capsize=5,
        label="Faucher-Giguère et al. (2008)",
        color='C2',
        barsabove=True,
        elinewidth=2,
        markeredgewidth=2,
        zorder=3,
    )

    ax.set_xlabel("Redshift")
    ax.set_ylabel("Mean Flux")
    ax.set_ylim([0.5, 1.1])

    bar_centers = [zhi for zlo, zhi in result["z_edges"]]
    ax2 = ax.twinx()
    ax2.bar(
        bar_centers, result["n_spectra"],
        zorder=1,
        width=0.1,
        alpha=0.5,
        color="gray"
    )
    ax2.set_ylabel("Number of usable Spectra")

    # Put the bar axis behind the main axis
    ax2.set_zorder(0)
    ax.set_zorder(1)
    ax.patch.set_visible(False)

    # Secondary axis: wavelength
    def redshift_to_wavelength(z):
        return 1215.67 * (z + 1)

    def wavelength_to_redshift(wavelength):
        return wavelength / 1215.67 - 1

    secax = ax.secondary_xaxis('top', functions=(redshift_to_wavelength, wavelength_to_redshift))
    secax.set_xlabel(r' Wavelength ($\AA$)')

    ax.legend(loc="lower left")

    if save:
        plt.savefig("plots/z_vs_mean_flux.pdf", format='pdf', bbox_inches='tight')
    plt.show()



def find_extreme_spectra(data, z_edges, bin_index, min_pix=5, n_examples=5, mode="both"):
    """
    Finds the spectra with the most extreme per-spectrum mean flux in a given
    redshift bin, for visual inspection.

    Args:
        data: dict from load_spectra_cache
        z_edges: list of (z_low, z_high) tuples (same bins used elsewhere)
        bin_index: which bin (index into z_edges) to inspect
        min_pix: minimum good pixels required for a spectrum to qualify (same
                 cut used in mean_flux_in_zbins, for consistency)
        n_examples: how many extreme spectra to return per side
        mode: "high" (most positive means), "low" (most negative means),
              or "both" (n_examples from each side)

    Returns:
        dict with:
            'indices': array of spectrum indices into data['flux'], sorted by
                       how extreme they are (only present if mode='high' or 'low')
            'high_indices', 'low_indices': present if mode='both'
            'per_spectrum_mean': the full per-spectrum mean array for this bin
                                  (NaN for non-qualifying spectra), useful for
                                  histograms or further inspection
            'qualifies': boolean mask of which spectra qualified for this bin
    """
    wavelength = data["wavelength"]
    flux = data["flux"]
    mask = data["mask"]
    z_qso = data["redshift"]

    zlo, zhi = z_edges[bin_index]
    wlo = LYA_REST * (1 + zlo)
    whi = LYA_REST * (1 + zhi)
    col_mask = (wavelength >= wlo) & (wavelength < whi)

    sub_flux = flux[:, col_mask]
    sub_mask = mask[:, col_mask]

    z_qualifies = z_qso > zhi
    counts_per_spectrum = np.sum(sub_mask, axis=1)
    qualifies = (counts_per_spectrum >= min_pix) & z_qualifies

    n = flux.shape[0]
    per_spectrum_mean = np.full(n, np.nan)
    qualifying_idx = np.where(qualifies)[0]
    for i in qualifying_idx:
        per_spectrum_mean[i] = np.mean(sub_flux[i][sub_mask[i]])

    result = {
        "per_spectrum_mean": per_spectrum_mean,
        "qualifies": qualifies,
        "bin": (zlo, zhi),
    }

    valid_means = per_spectrum_mean[qualifying_idx]
    order = np.argsort(valid_means)  # ascending

    if mode in ("high", "both"):
        high_idx = qualifying_idx[order[-n_examples:]][::-1]  # highest first
        result["high_indices"] = high_idx
    if mode in ("low", "both"):
        low_idx = qualifying_idx[order[:n_examples]]  # lowest first
        result["low_indices"] = low_idx
    if mode not in ("high", "low", "both"):
        raise ValueError("mode must be 'high', 'low', or 'both'")

    return result

In [ ]:
z_edges = [(2.00, 2.05), (2.10, 2.15), (2.20, 2.25)]

extremes = find_extreme_spectra(data, z_edges, bin_index=0, min_pix=5, n_examples=5, mode="both")

print("Highest mean-flux spectra:", extremes["high_indices"],
      "means:", extremes["per_spectrum_mean"][extremes["high_indices"]])
print("Lowest mean-flux spectra:", extremes["low_indices"],
      "means:", extremes["per_spectrum_mean"][extremes["low_indices"]])


In [ ]:
spectrum_idx = 0

mask = (data["wavelength"] >= 3600) & (data["wavelength"] <= 4000)

print(mask)

fig, ax1 = plt.subplots(figsize=(10, 3))

ax1.plot(data["wavelength"][mask], data["flux"][spectrum_idx][mask], color="C0")
ax1.set_xlabel("Wavelength (Å)")
ax1.set_ylabel("Flux", color="C0")

ax2 = ax1.twinx()
ax2.plot(data["wavelength"][mask], data["snr"][spectrum_idx][mask], color="C1")
ax2.set_ylabel("SNR", color="C1")

print(np.nanmedian(data["snr"][spectrum_idx][mask]))

plt.show()

In [ ]:
def spectrum_snr_mask(sub_snr, sub_mask, min_snr=1.0):
    """Returns a boolean array (per spectrum) of whether the spectrum's median
    S/N in this bin is above threshold. Assumes sub_snr IS the S/N ratio directly."""
    n = sub_snr.shape[0]
    keep = np.zeros(n, dtype=bool)
    for i in range(n):
        m = sub_mask[i]
        if not np.any(m):
            continue
        keep[i] = np.nanmedian(sub_snr[i][m]) >= min_snr

    # print(f"Filtered out {n - np.nansum(keep)} Spectra because their snr < 1")
    return keep


def mean_flux_in_zbins(data, z_edges, min_pix=5, min_snr=1.0):
    """
    Computes the pixel-weighted mean flux (and its uncertainty) in each given
    Lyman-alpha absorption redshift bin, using the cached common wavelength grid.

    This version treats data["snr"] as the signal-to-noise ratio directly
    (F / sigma_F), rather than sigma_F itself.

    Args:
        data: dict from load_spectra_cache
        z_edges: list of (z_low, z_high) tuples
        min_pix: minimum number of good pixels a spectrum must contribute to a bin
        min_snr: minimum median S/N a spectrum must have in the bin
    """
    wavelength = data["wavelength"]
    flux = data["flux"]
    mask = data["mask"]
    snr = data["snr"]  # now treated as F/sigma_F directly
    z_qso = data["redshift"]

    n_bins = len(z_edges)
    mean_flux = np.full(n_bins, np.nan)
    sem = np.full(n_bins, np.nan)
    n_pixels_per_bin = np.zeros(n_bins, dtype=int)
    n_spectra_per_bin = np.zeros(n_bins, dtype=int)

    for b, (zlo, zhi) in enumerate(z_edges):

        wlo = LYA_REST * (1 + zlo)
        whi = LYA_REST * (1 + zhi)
        col_mask = (wavelength >= wlo) & (wavelength < whi)

        sub_flux = flux[:, col_mask]
        sub_mask = mask[:, col_mask]
        sub_snr = snr[:, col_mask]

        z_qualifies = z_qso > zhi
        snr_ok = spectrum_snr_mask(sub_snr, sub_mask, min_snr=min_snr)
        counts_per_spectrum = np.sum(sub_mask, axis=1)

        qualifies = (counts_per_spectrum >= min_pix) & z_qualifies & snr_ok

        qualifying_flux = sub_flux[qualifies]
        qualifying_mask = sub_mask[qualifies]

        per_spectrum_mean = np.full(qualifying_flux.shape[0], np.nan)
        for i in range(qualifying_flux.shape[0]):
            per_spectrum_mean[i] = np.mean(qualifying_flux[i][qualifying_mask[i]])

        if per_spectrum_mean.size > 0:
            all_vals = qualifying_flux[qualifying_mask]
            mean_flux[b] = np.mean(all_vals)
            n_pixels_per_bin[b] = all_vals.size

            n_spec = per_spectrum_mean.size
            sem[b] = np.std(per_spectrum_mean, ddof=1) # / np.sqrt(n_spec)
            n_spectra_per_bin[b] = n_spec

    return {
        "z_edges": z_edges,
        "mean_flux": mean_flux,
        "sem": sem,
        "n_pixels": n_pixels_per_bin,
        "n_spectra": n_spectra_per_bin,
    }

In [ ]:
result = mean_flux_in_zbins(data, redshift_bins, min_pix=20)
plot_mean_flux(result, F, z_paper, dF_minus, dF_plus, redshifts_sims, mean_fluxes_sims, std_fluxes_sims)
print(result["mean_flux"])
print(result["sem"])
print(result["n_spectra"])

In [ ]:
def mean_flux_in_zbins(data, z_edges, min_pix=5, min_snr=1.0):
    """
    Computes the pixel-weighted mean flux (and its uncertainty) in each given
    Lyman-alpha absorption redshift bin, using the cached common wavelength grid.

    Spectrum-level S/N filtering is done ONCE over the full cropped spectrum
    (not per-bin): a spectrum is entirely excluded from all bins if its overall
    median S/N (over all good pixels across the whole wavelength range) is
    below min_snr.

    Args:
        data: dict from load_spectra_cache
        z_edges: list of (z_low, z_high) tuples
        min_pix: minimum number of good pixels a spectrum must contribute to a
                 given bin to be counted
        min_snr: minimum median S/N (over the WHOLE spectrum) required for a
                 spectrum to be used at all
    """
    wavelength = data["wavelength"]
    flux = data["flux"]
    mask = data["mask"]
    snr = data["snr"]  # F/sigma_F directly
    z_qso = data["redshift"]

    # -------- Spectrum-level S/N filter, computed ONCE over the full spectrum --------
    n_spectra_total = flux.shape[0]
    spectrum_snr_ok = np.zeros(n_spectra_total, dtype=bool)
    for i in range(n_spectra_total):
        m = mask[i]
        if not np.any(m):
            continue
        spectrum_snr_ok[i] = np.nanmin(snr[i][m]) >= min_snr
    # -----------------------------------------------------------------------------

    n_bins = len(z_edges)
    mean_flux = np.full(n_bins, np.nan)
    sem = np.full(n_bins, np.nan)
    n_pixels_per_bin = np.zeros(n_bins, dtype=int)
    n_spectra_per_bin = np.zeros(n_bins, dtype=int)

    raw_means = []

    for b, (zlo, zhi) in enumerate(z_edges):
        wlo = LYA_REST * (1 + zlo)
        whi = LYA_REST * (1 + zhi)
        col_mask = (wavelength >= wlo) & (wavelength < whi)

        sub_flux = flux[:, col_mask]
        sub_mask = mask[:, col_mask]

        z_qualifies = z_qso > zhi
        counts_per_spectrum = np.sum(sub_mask, axis=1)

        # Now just AND in the pre-computed whole-spectrum S/N filter
        qualifies = (counts_per_spectrum >= min_pix) & z_qualifies & spectrum_snr_ok

        qualifying_flux = sub_flux[qualifies]
        qualifying_mask = sub_mask[qualifies]

        raw_means.append([])

        per_spectrum_mean = np.full(qualifying_flux.shape[0], np.nan)
        for i in range(qualifying_flux.shape[0]):
            per_spectrum_mean[i] = np.mean(qualifying_flux[i][qualifying_mask[i]])
            raw_means[b].append(per_spectrum_mean[i])

        if per_spectrum_mean.size > 0:
            all_vals = qualifying_flux[qualifying_mask]
            mean_flux[b] = np.mean(all_vals)
            n_pixels_per_bin[b] = all_vals.size

            n_spec = per_spectrum_mean.size
            sem[b] = np.std(per_spectrum_mean, ddof=1)
            n_spectra_per_bin[b] = n_spec

    return {
        "raw_means": raw_means,
        "z_edges": z_edges,
        "mean_flux": mean_flux,
        "sem": sem,
        "n_pixels": n_pixels_per_bin,
        "n_spectra": n_spectra_per_bin,
    }

In [ ]:
result = mean_flux_in_zbins(data, redshift_bins, min_pix=20)
plot_mean_flux(result, F, z_paper, dF_minus, dF_plus, redshifts_sims, mean_fluxes_sims, std_fluxes_sims, save=True)
print(result["mean_flux"])
print(result["sem"])
print(result["n_spectra"])
print(result["n_pixels"])

## Make plot of actual distributions

In [ ]:
fluxes_plot = []

for i, z in enumerate(redshifts_sims):
    w, y = load_spectra_from_boxes(gps_to_reorder, spectrumNums=[i for i in range(10000)], z=z)
    fluxes = np.array(y[0])
    l, r = get_edges(gps_to_reorder[0], snapN=snaps[i], redshift=z)
    fluxes = fluxes[:, (w[0] >= l) & (w[0] <= r)]
    fluxes_plot.append(fluxes)

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

range_flux = (0.3, 1.0)

colors = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
    "#17becf",
    "#4c78a4",
]

n = len(redshifts_sims)

fig, axes = plt.subplots(
    3, 4,
    figsize=(12, 8),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for i in range(n):
    ax = axes[i]

    # Filled histogram — SDSS data
    h1, edges1 = np.histogram(
        result["raw_means"][i],
        bins=50,
        range=range_flux
    )

    ax.stairs(
        h1,
        edges1,
        fill=True,
        alpha=0.6,
        color=colors[i],
        label="SDSS data"
    )

    # Outline histogram — simulated data
    h2, edges2 = np.histogram(
        fluxes_plot[i].mean(axis=1),
        bins=50,
        range=range_flux
    )

    ax.stairs(
        h2,
        edges2,
        fill=False,
        linewidth=2,
        color=colors[i],
        label="sim data"
    )

    # Literature confidence interval
    ax.axvspan(
        F_low[i],
        F_high[i],
        color="black",
        alpha=0.15,
        zorder=0
    )

    # Literature reference value
    ax.axvline(
        F[i],
        color="black",
        linestyle="--",
        linewidth=2,
        zorder=5,
        label="literature"
    )

    ax.legend(fontsize=10)
    ax.set_title(f"z = {redshifts_sims[i]:.1f}")
    ax.set_xlim(range_flux)


# --------------------------------------------------
# Make all panels use the same y-axis range
# --------------------------------------------------

ymax = max(ax.get_ylim()[1] for ax in axes[:n])

for ax in axes[:n]:
    ax.set_ylim(0, ymax)


# --------------------------------------------------
# Legend handles
# --------------------------------------------------

sdss_handle = Patch(
    facecolor="gray",
    edgecolor="gray",
    alpha=0.6,
    label="SDSS data"
)

sim_handle = Line2D(
    [0], [0],
    color="gray",
    linewidth=2,
    label="Simulated data"
)

literature_handle = Line2D(
    [0], [0],
    color="black",
    linestyle="--",
    linewidth=2,
    label="Literature"
)

confidence_handle = Patch(
    facecolor="black",
    edgecolor="none",
    alpha=0.15,
    label="Literature CI"
)


# --------------------------------------------------
# Remove unused 12th panel
# --------------------------------------------------

for i in range(n, len(axes)):
    axes[i].set_visible(False)


# Labels
fig.supxlabel("Mean Flux per spectrum")
fig.supylabel("Number of spectra")

plt.tight_layout()
plt.savefig("plots/mean_flux_distributions.pdf", format='pdf', bbox_inches='tight')
plt.show()
